# CodeGen Capstone — Week 1 baselines + Mentor dataset extension

**Self-contained notebook** — all helpers are defined inline (no git clone, no `lib/` imports).

| Part | What it does |
|------|----------------|
| **1** | Setup, inline library, Java, GPU |
| **2** | K=0 baselines (Qwen 1.5B vs 7B) |
| **3** | Mentor task: extend datasets (NL + PL1 + PL2) |

**Before running:** Runtime → **T4 GPU**


## Part 1 — Setup + inline library (run once per Colab session)


In [ ]:
# 1.1 — mount Drive and set paths
import os

from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

PROJECT_DIR = os.environ.get('CODEGEN_PROJECT_DIR', '/content/drive/MyDrive/codegen_week1')
os.makedirs(PROJECT_DIR, exist_ok=True)
os.environ['CODEGEN_PROJECT_DIR'] = PROJECT_DIR
os.environ['CODEGEN_DATA_DIR']    = PROJECT_DIR

%cd {PROJECT_DIR}
print('Project dir:', os.getcwd())


In [ ]:
# 1.2 — install OpenJDK 17
!apt-get update -qq
!apt-get install -y -q openjdk-17-jdk-headless > /dev/null
!java -version
!javac -version


In [ ]:
# 1.3 — install Python deps
!pip install -q \
  'transformers>=4.45.0' 'accelerate>=0.34.0' 'bitsandbytes>=0.43.0' \
  'datasets>=2.20.0' 'sentencepiece>=0.2.0' 'huggingface-hub>=0.24.0' \
  'numpy>=1.26.0' 'pandas>=2.1.0' 'tqdm>=4.66.0'


In [ ]:
# 1.4 — GPU check
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Runtime -> Change runtime type -> T4 GPU.')
p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name}, {p.total_memory/1e9:.1f} GB')


### 1.5 — Inline `config`


In [ ]:
from __future__ import annotations
import os
from pathlib import Path

# ---------------- paths ----------------
#  base data dir; benchmarks cache; tools (JUnit JAR); results CSV target
DATA_DIR = Path(os.environ.get("CODEGEN_DATA_DIR", "/content/drive/MyDrive/codegen_task3_v4"))
BENCHMARKS_DIR = DATA_DIR / "benchmarks"          # 
TOOLS_DIR      = DATA_DIR / "tools"               #   (JUnit5 standalone JAR)
RESULTS_DIR    = DATA_DIR / "results"             #   (baselines.csv lives here)


#  JUnit5 console-launcher (downloaded once by setup_java_runtime, used by run_java)
JUNIT_JAR_VERSION = "1.10.1"
JUNIT_JAR = TOOLS_DIR / f"junit-platform-console-standalone-{JUNIT_JAR_VERSION}.jar"
JUNIT_JAR_URL = (
    f"https://repo1.maven.org/maven2/org/junit/platform/"
    f"junit-platform-console-standalone/{JUNIT_JAR_VERSION}/"
    f"junit-platform-console-standalone-{JUNIT_JAR_VERSION}.jar"
)
# ---------------- models ----------------

QWEN_1_5B = "Qwen/Qwen2.5-Coder-1.5B-Instruct"  
QWEN_7B   = "Qwen/Qwen2.5-Coder-7B-Instruct"  

MODELS = {"1.5b": QWEN_1_5B, "7b": QWEN_7B}     

# ---------------- run constants ----------------
SEED = int(os.environ.get("CODEGEN_SEED", "42"))                                  



#  decoding for K=0 baselines (greedy). DECODING_SAMPLED is for pass@10 (cp3 sampling).
DECODING_GREEDY  = {"do_sample": False, "temperature": 0.0, "max_new_tokens": 512}    

RETRIEVAL_TOKEN_CAP = 8000                                                        

# subprocess timeout for Python and Java sandboxes
EXEC_TIMEOUT_S = 10                                                               

#   CODEGEN_SMOKE=1   -> 20 problems per benchmark (pipeline smoke test)          
#   CODEGEN_SAMPLE=N  -> N problems per benchmark (reproducible random subset)    
#   neither           -> full benchmarks (164 / 257 / 164 / ...)                  
SMOKE = os.environ.get("CODEGEN_SMOKE", "0") == "1"                               
_SAMPLE_RAW = os.environ.get("CODEGEN_SAMPLE", "")
SAMPLE_SIZE = int(_SAMPLE_RAW) if _SAMPLE_RAW.isdigit() and int(_SAMPLE_RAW) > 0 else None  

BENCHMARK_LIMIT = 20 if SMOKE else SAMPLE_SIZE  # None = all problems             

# ---------------- dataset extension (mentor task) ----------------
EXTENDED_DIR = DATA_DIR / "extended"
EXTEND_MAX_RETRIES = int(os.environ.get("CODEGEN_EXTEND_RETRIES", "3"))
# Use CODEGEN_EXTEND_SAMPLE=N to cap extension rows (defaults to BENCHMARK_LIMIT)
_EXTEND_RAW = os.environ.get("CODEGEN_EXTEND_SAMPLE", "")
EXTEND_LIMIT = int(_EXTEND_RAW) if _EXTEND_RAW.isdigit() and int(_EXTEND_RAW) > 0 else BENCHMARK_LIMIT


def ensure_dirs() -> None:   
    for d in [DATA_DIR, BENCHMARKS_DIR, TOOLS_DIR, RESULTS_DIR, EXTENDED_DIR]:
        d.mkdir(parents=True, exist_ok=True)


import types
config = types.ModuleType('config')
for _name in (
    'DATA_DIR', 'BENCHMARKS_DIR', 'TOOLS_DIR', 'RESULTS_DIR', 'EXTENDED_DIR',
    'JUNIT_JAR_VERSION', 'JUNIT_JAR', 'JUNIT_JAR_URL',
    'QWEN_1_5B', 'QWEN_7B', 'MODELS', 'SEED', 'DECODING_GREEDY', 'RETRIEVAL_TOKEN_CAP',
    'EXEC_TIMEOUT_S', 'SMOKE', 'SAMPLE_SIZE', 'BENCHMARK_LIMIT',
    'EXTEND_MAX_RETRIES', 'EXTEND_LIMIT', 'ensure_dirs', 'refresh_config_limits',
):
    setattr(config, _name, globals()[_name])

def refresh_config_limits():
    import os
    global SMOKE, SAMPLE_SIZE, BENCHMARK_LIMIT, EXTEND_MAX_RETRIES, EXTEND_LIMIT
    SMOKE = os.environ.get('CODEGEN_SMOKE', '0') == '1'
    _sample = os.environ.get('CODEGEN_SAMPLE', '')
    SAMPLE_SIZE = int(_sample) if _sample.isdigit() and int(_sample) > 0 else None
    BENCHMARK_LIMIT = 20 if SMOKE else SAMPLE_SIZE
    EXTEND_MAX_RETRIES = int(os.environ.get('CODEGEN_EXTEND_RETRIES', '3'))
    _ext = os.environ.get('CODEGEN_EXTEND_SAMPLE', '')
    EXTEND_LIMIT = int(_ext) if _ext.isdigit() and int(_ext) > 0 else BENCHMARK_LIMIT
    for _name in ('SMOKE', 'SAMPLE_SIZE', 'BENCHMARK_LIMIT', 'EXTEND_MAX_RETRIES', 'EXTEND_LIMIT'):
        setattr(config, _name, globals()[_name])

config.refresh_config_limits = refresh_config_limits
ensure_dirs()
print('Data dir:', config.DATA_DIR)


### 1.6 — Inline execution sandbox


In [ ]:
from __future__ import annotations
import os
import re
import subprocess
import sys
import tempfile
from pathlib import Path



# ---------------- Python ----------------
def run_python(full_program: str, timeout: int = config.EXEC_TIMEOUT_S) -> dict:
    """`full_program` should already include the test harness (assertions, calls, etc).
    Returns {"passed": bool, "error": str | None}.
    """
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / "solution.py"
        path.write_text(full_program, encoding="utf-8")
        try:
            result = subprocess.run(
                [sys.executable, str(path)],
                capture_output=True, text=True, timeout=timeout,
                env={**os.environ, "PYTHONDONTWRITEBYTECODE": "1"},
            )
            if result.returncode == 0:
                return {"passed": True, "error": None}
            err = (result.stderr or result.stdout).strip().splitlines()[-1:]
            return {"passed": False, "error": "\n".join(err) or "nonzero exit"}
        except subprocess.TimeoutExpired:
            return {"passed": False, "error": f"timeout >{timeout}s"}
        except Exception as e:
            return {"passed": False, "error": f"runner: {e!r}"}


# ---------------- Java ----------------
_CLASS_NAME_RE = re.compile(r"public\s+class\s+(\w+)")

# Imports the original HumanEval-X scoring expects to be available in BOTH
# Solution.java and Main.java. The reference scoring uses a single combined
# file (prompt + canonical_solution + test), where the prompt's imports flow
# through. Since we split into two files, each file needs its own import
# section. Java tolerates duplicate imports (just a warning), so prepending
# unconditionally is safe.
_JAVA_STD_IMPORTS = (
    "import java.util.*;\n"
    "import java.util.stream.*;\n"
    "import java.util.regex.*;\n"
    "import java.lang.*;\n"
    "import java.math.*;\n"
)


def _extract_classes(java_src: str) -> list[str]:
    return _CLASS_NAME_RE.findall(java_src)


def _inject_imports(src: str) -> str:
    """Prepend standard java.util / java.util.stream imports if not already at top."""
    head = src.lstrip()
    if head.startswith("package "):  # never seen for HumanEval-X but defensive
        nl = head.find("\n")
        return head[: nl + 1] + _JAVA_STD_IMPORTS + head[nl + 1 :]
    return _JAVA_STD_IMPORTS + src


def run_java(java_source: str, test_source: str, timeout: int = config.EXEC_TIMEOUT_S) -> dict:
    """Compile both files, run the test class' main() (HumanEval-X convention).

    java_source: the model's translated code (typically class Solution { ... })
    test_source: the reference test file from HumanEval-X (typically class Main { public static void main ... })
    Returns {"passed": bool, "error": str | None}.
    """
    classes_in_test = _extract_classes(test_source)
    main_class = classes_in_test[0] if classes_in_test else "Main"

    # HumanEval-X test files reference List/Arrays/Collectors etc. without imports;
    # inject standard imports to make the split-file harness compile.
    java_source = _inject_imports(java_source)
    test_source = _inject_imports(test_source)

    with tempfile.TemporaryDirectory() as tmp:
        tmp_path = Path(tmp)
        # split combined java_source into per-class files (compiler tolerates one-file multi-class
        # but write_text once is simplest)
        sol_path = tmp_path / "Solution.java"
        test_path = tmp_path / f"{main_class}.java"
        sol_path.write_text(java_source, encoding="utf-8")
        test_path.write_text(test_source, encoding="utf-8")

        try:
            compile_res = subprocess.run(
                ["javac", "-d", str(tmp_path), str(sol_path), str(test_path)],
                capture_output=True, text=True, timeout=timeout,
            )
            if compile_res.returncode != 0:
                return {"passed": False, "error": "compile: " + (compile_res.stderr or "").strip()[:300]}

            run_res = subprocess.run(
                ["java", "-cp", str(tmp_path), main_class],
                capture_output=True, text=True, timeout=timeout,
            )
            if run_res.returncode == 0:
                return {"passed": True, "error": None}
            tail = (run_res.stderr or run_res.stdout).strip().splitlines()[-3:]
            return {"passed": False, "error": "\n".join(tail) or "nonzero exit"}
        except subprocess.TimeoutExpired:
            return {"passed": False, "error": f"timeout >{timeout}s"}
        except Exception as e:
            return {"passed": False, "error": f"runner: {e!r}"}


def compile_java(java_source: str, timeout: int = config.EXEC_TIMEOUT_S) -> dict:
    """Compile Java source only (no test execution)."""
    with tempfile.TemporaryDirectory() as tmp:
        tmp_path = Path(tmp)
        sol_path = tmp_path / "Solution.java"
        sol_path.write_text(_inject_imports(java_source), encoding="utf-8")
        try:
            compile_res = subprocess.run(
                ["javac", str(sol_path)],
                capture_output=True, text=True, timeout=timeout,
            )
            if compile_res.returncode == 0:
                return {"passed": True, "error": None}
            return {"passed": False, "error": "compile: " + (compile_res.stderr or "").strip()[:300]}
        except subprocess.TimeoutExpired:
            return {"passed": False, "error": f"timeout >{timeout}s"}
        except Exception as e:
            return {"passed": False, "error": f"runner: {e!r}"}


def smoke_test_java() -> dict:
    """Sanity-check the Java toolchain. Compiles and runs a trivial passing program."""
    sol = "public class Solution { public static int add(int a, int b) { return a + b; } }"
    tst = (
        "public class Main {\n"
        "    public static void main(String[] args) {\n"
        "        if (Solution.add(2, 3) != 5) throw new AssertionError();\n"
        "    }\n"
        "}\n"
    )
    return run_java(sol, tst)


### 1.7 — Inline prompts + generation


In [ ]:
from __future__ import annotations
import re

def _apply_chat(tokenizer, system: str, user: str) -> str:
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


# ----------------- HumanEval (Python completion) -----------------
def build_completion_prompt(tokenizer, problem: dict) -> str:
    """HumanEval-style code completion.

    The model is expected to return the COMPLETE Python program (imports +
    function definition with body) inside a ```python ... ``` block. The
    scorer treats the extracted block as the entire program — no concatenation
    with the original prompt.
    """
    system = "You are an expert Python programmer."
    user = (
        f"Complete the following Python code. "
        f"Return the COMPLETE program (imports, the full function signature, and the implementation) "
        f"in a single ```python ... ``` code block. "
        f"Do NOT return only the body — repeat the function signature and any required imports.\n\n"
        f"```python\n{problem['prompt']}\n```"
    )
    return _apply_chat(tokenizer, system, user)


# ----------------- HumanEval-X (Python -> Java translation) -----------------
def build_translation_prompt(tokenizer, python_source: str, java_declaration: str) -> str:
    """Python -> Java translation.

    The model is expected to return a COMPLETE Java solution — imports +
    `class Solution { ... }` with the method — inside a ```java ... ``` block.
    The scorer writes the extracted code to `Solution.java` and compiles it
    together with the reference test (a separate `class Main`). NOTE: do NOT
    prepend `java_declaration`; the model's output is the entire Solution.java.
    `java_declaration` is shown only for signature reference.
    """
    system = "You are a code translator. You translate Python to Java preserving behavior."
    user = (
        f"Translate the following Python function to Java. Preserve behavior exactly.\n"
        f"Return the COMPLETE Java solution — imports + `class Solution {{ ... }}` containing "
        f"the translated method — inside a single ```java ... ``` code block. "
        f"Match the method signature shown below.\n\n"
        f"### Python source\n```python\n{python_source}\n```\n\n"
        f"### Required Java method signature\n```java\n{java_declaration}\n```"
    )
    return _apply_chat(tokenizer, system, user)


# ----------------- generic generate -----------------
def generate(tokenizer, model, prompt: str, decoding: dict) -> str:
    import torch
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            **decoding,
        )
    text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return text


# ----------------- code-fence-aware extractors -----------------
def _strip_fences(text: str, lang_hints: tuple[str, ...]) -> str:
    """Return the contents of the FIRST ```<lang>?\\n ... ``` block in `text`,
    falling back to the stripped text if no fence is present.

    Tolerates a missing closing fence (returns from the open fence to end of
    text). Does NOT strip language tokens like `def` or `class` from the body —
    that's the caller's job (and our scorers no longer do it).
    """
    s = text.strip()
    open_re = re.compile(r"```(?:" + "|".join(lang_hints) + r")?\s*\n?", re.IGNORECASE)
    m = open_re.search(s)
    if not m:
        return (s + "\n") if s else ""
    body = s[m.end():]
    close = re.search(r"```", body)
    body = body[:close.start()] if close else body
    body = body.rstrip()
    return body + "\n" if body else ""


def extract_python_body(text: str) -> str:
    """Extract Python code from a model response. Returns the contents of the
    first ```python ... ``` (or unlabeled ``` ... ```) block. If no fence is
    found, returns the whole stripped text."""
    return _strip_fences(text, ("python", "py"))


# ----------------- Mentor extension: PL2 from NL+PL1 -----------------
def build_pl2_generation_prompt(tokenizer, nl: str, pl1: str, feedback: str | None = None) -> str:
    system = "You are an expert Java programmer."
    fb = f"\n\n### Previous attempt feedback\n{feedback}\n" if feedback else ""
    user = (
        "Implement the following specification in Java.\n"
        "You are given the English description (NL) and a reference Python solution (PL1).\n"
        "Return the COMPLETE Java solution — imports + `class Solution { ... }` — "
        "in a single ```java ... ``` code block. Preserve behavior.\n"
        f"{fb}\n"
        f"### English description (NL)\n{nl}\n\n"
        f"### Reference Python (PL1)\n```python\n{pl1}\n```"
    )
    return _apply_chat(tokenizer, system, user)


def build_pl2_feedback_prompt(
    tokenizer, nl: str, pl1: str, pl2_attempt: str, discrepancy: str
) -> str:
    return build_pl2_generation_prompt(
        tokenizer, nl, pl1,
        feedback=(
            f"The previous Java (PL2) did not validate.\n"
            f"Discrepancy: {discrepancy}\n\n"
            f"Previous PL2:\n```java\n{pl2_attempt}\n```\n"
            "Generate a corrected PL2."
        ),
    )


# ----------------- Mentor extension: NL from PL1+PL2 -----------------
def build_nl_generation_prompt(tokenizer, pl1: str, pl2: str, feedback: str | None = None) -> str:
    system = "You are an expert at describing code in clear English."
    fb = f"\n\n### Previous attempt feedback\n{feedback}\n" if feedback else ""
    user = (
        "Write a precise English description of what this program does.\n"
        "The description must be detailed enough that a developer could re-implement "
        "the solution in Python or Java from your text alone.\n"
        "Return ONLY the description (no code fences).\n"
        f"{fb}\n"
        f"### Python (PL1)\n```python\n{pl1}\n```\n\n"
        f"### Java (PL2)\n```java\n{pl2}\n```"
    )
    return _apply_chat(tokenizer, system, user)


def build_nl_feedback_prompt(
    tokenizer, nl_attempt: str, pl1: str, pl2: str, discrepancy: str
) -> str:
    return build_nl_generation_prompt(
        tokenizer, pl1, pl2,
        feedback=(
            f"The current description failed validation when used to regenerate code.\n"
            f"Discrepancy: {discrepancy}\n\n"
            f"Current NL:\n{nl_attempt}\n\n"
            "Update the English description to fix the discrepancy."
        ),
    )


def build_java_completion_from_nl_prompt(tokenizer, java_prompt: str) -> str:
    """NL → PL2 direct completion (HumanEval-X java_prompt = signature + docstring)."""
    system = "You are an expert Java programmer."
    user = (
        "Complete the following Java code from the English specification.\n"
        "Return the COMPLETE program — imports + `class Solution { ... }` — "
        "in a single ```java ... ``` code block.\n\n"
        f"```java\n{java_prompt}\n```"
    )
    return _apply_chat(tokenizer, system, user)


def extract_java_body(text: str) -> str:
    """Extract Java code from a model response. Returns the contents of the
    first ```java ... ``` (or unlabeled ``` ... ```) block. If no fence is
    found, returns the whole stripped text."""
    return _strip_fences(text, ("java",))


### 1.8 — Inline model loaders


In [ ]:
from __future__ import annotations
import gc
import urllib.request
from pathlib import Path
from typing import Tuple

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig



def _bnb_4bit_config() -> BitsAndBytesConfig:
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )


def load_qwen(size: str) -> Tuple[AutoTokenizer, AutoModelForCausalLM]:
    """size in {'1.5b', '7b'}. Returns (tokenizer, model) on GPU."""
    model_id = config.MODELS[size]
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    if size == "7b":
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=_bnb_4bit_config(),
            device_map="auto",
            trust_remote_code=True,
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
        )
    model.eval()
    return tokenizer, model


def unload(*_unused) -> None:  
    gc.collect()
    torch.cuda.empty_cache()


def setup_java_runtime() -> None:
    """Idempotent: download JUnit5 console-launcher JAR if missing."""
    config.ensure_dirs()
    jar = Path(config.JUNIT_JAR)
    if jar.exists():
        return
    print(f"Downloading {config.JUNIT_JAR_URL} ...")
    urllib.request.urlretrieve(config.JUNIT_JAR_URL, jar)
    assert jar.exists(), "JUnit JAR download failed"
    print(f"Saved to {jar} ({jar.stat().st_size / 1e6:.1f} MB)")


### 1.9 — Inline AST / validation helpers


In [ ]:
from __future__ import annotations

import ast
import re



def extract_docstring_from_prompt(prompt: str) -> str:
    """Pull the triple-quoted docstring from a HumanEval-style prompt."""
    m = re.search(r'"""(.*?)"""', prompt, re.DOTALL)
    if m:
        return m.group(1).strip()
    m = re.search(r"'''(.*?)'''", prompt, re.DOTALL)
    if m:
        return m.group(1).strip()
    return prompt.strip()


def python_ast_dump(source: str) -> str | None:
    try:
        tree = ast.parse(source)
        return ast.dump(tree, annotate_fields=False)
    except SyntaxError:
        return None


def python_ast_match(code_a: str, code_b: str) -> tuple[bool, str | None]:
    """Return (match, error_message)."""
    a = python_ast_dump(code_a)
    b = python_ast_dump(code_b)
    if a is None:
        return False, "code_a: syntax error"
    if b is None:
        return False, "code_b: syntax error"
    if a == b:
        return True, None
    return False, "python AST mismatch"


def python_passes_humaneval(program: str, test: str, entry_point: str) -> dict:
    full = program.rstrip() + "\n\n" + test + f"\n\ncheck({entry_point})\n"
    return run_python(full, timeout=config.EXEC_TIMEOUT_S)


def python_passes_mbpp(program: str, tests: list[str]) -> dict:
    full = program.rstrip() + "\n\n" + "\n".join(tests) + "\n"
    return run_python(full, timeout=config.EXEC_TIMEOUT_S)


def python_output_match_humaneval(code_a: str, code_b: str, test: str, entry_point: str) -> tuple[bool, str | None]:
    """Both programs must pass the same HumanEval harness (proxy for output equivalence)."""
    ra = python_passes_humaneval(code_a, test, entry_point)
    rb = python_passes_humaneval(code_b, test, entry_point)
    if ra["passed"] and rb["passed"]:
        return True, None
    parts = []
    if not ra["passed"]:
        parts.append(f"code_a: {ra['error']}")
    if not rb["passed"]:
        parts.append(f"code_b: {rb['error']}")
    return False, "; ".join(parts) or "output mismatch"


def python_output_match_mbpp(code_a: str, code_b: str, tests: list[str]) -> tuple[bool, str | None]:
    ra = python_passes_mbpp(code_a, tests)
    rb = python_passes_mbpp(code_b, tests)
    if ra["passed"] and rb["passed"]:
        return True, None
    parts = []
    if not ra["passed"]:
        parts.append(f"code_a: {ra['error']}")
    if not rb["passed"]:
        parts.append(f"code_b: {rb['error']}")
    return False, "; ".join(parts) or "output mismatch"


def validate_pl2_against_pl1_humaneval(
    pl1: str,
    pl2: str,
    *,
    test: str,
    entry_point: str,
    java_test: str | None,
) -> dict:
    """Mentor validation for generated PL2.

    Output: PL1 passes Python tests AND PL2 passes Java tests (same problem).
    AST: PL1 reference must parse (structural sanity on PL1).
    """
    py_ref = python_passes_humaneval(pl1, test, entry_point)
    if not py_ref["passed"]:
        return {
            "valid": False,
            "output_match": False,
            "ast_match": False,
            "java_pass": False,
            "error": f"pl1 reference failed tests: {py_ref['error']}",
        }

    ast_ok, ast_err = python_ast_match(pl1, pl1)
    if not ast_ok:
        return {"valid": False, "output_match": True, "ast_match": False, "java_pass": False, "error": ast_err}

    if not java_test:
        return {
            "valid": False,
            "output_match": True,
            "ast_match": True,
            "java_pass": False,
            "error": "no HumanEval-X java_test for this problem id",
        }

    jr = run_java(pl2, java_test)
    if not jr["passed"]:
        return {
            "valid": False,
            "output_match": True,
            "ast_match": True,
            "java_pass": False,
            "error": f"java tests failed: {jr['error']}",
        }

    return {"valid": True, "output_match": True, "ast_match": True, "java_pass": True, "error": None}


def validate_pl2_against_pl1_mbpp(pl1: str, pl2: str, tests: list[str]) -> dict:
    """MBPP has no Java tests: PL1 must pass asserts; PL2 must at least compile."""
    py_ref = python_passes_mbpp(pl1, tests)
    if not py_ref["passed"]:
        return {
            "valid": False,
            "output_match": False,
            "ast_match": False,
            "java_pass": False,
            "error": f"pl1 reference failed tests: {py_ref['error']}",
        }
    ast_ok, ast_err = python_ast_match(pl1, pl1)
    if not ast_ok:
        return {"valid": False, "output_match": True, "ast_match": False, "java_pass": False, "error": ast_err}
    jc = compile_java(pl2)
    if not jc["passed"]:
        return {
            "valid": False,
            "output_match": True,
            "ast_match": True,
            "java_pass": False,
            "error": f"pl2 compile failed: {jc['error']}",
        }
    return {
        "valid": True,
        "output_match": True,
        "ast_match": True,
        "java_pass": True,
        "error": None,
        "note": "MBPP: PL2 validated by compile only (no Java test harness)",
    }


def validate_nl_via_regeneration(
    nl: str,
    pl1_gt: str,
    pl2_gt: str,
    pl1_gen: str,
    pl2_gen: str,
    *,
    test: str | None,
    entry_point: str | None,
    java_test: str | None,
) -> dict:
    """Validate NL by checking regenerated PL1' and PL2' against ground truth."""
    pl1_out, pl1_out_err = python_output_match_humaneval(pl1_gt, pl1_gen, test, entry_point) if test and entry_point else (True, None)
    pl1_ast, pl1_ast_err = python_ast_match(pl1_gt, pl1_gen)

    pl2_java = run_java(pl2_gen, java_test) if java_test else {"passed": False, "error": "no java_test"}
    pl2_out_ok = pl2_java["passed"]
    pl2_ast, pl2_ast_err = (True, None)

    valid = pl1_out_ok and pl1_ast and pl2_out_ok
    errors = [e for e in (pl1_out_err, pl1_ast_err, pl2_java.get("error")) if e]
    return {
        "valid": valid,
        "pl1_output_match": pl1_out,
        "pl1_ast_match": pl1_ast,
        "pl2_java_pass": pl2_out_ok,
        "error": "; ".join(errors) if errors else None,
    }


### 1.10 — Inline benchmark loaders


In [ ]:

from __future__ import annotations
import gzip
import json
import random
from datasets import load_dataset
from huggingface_hub import hf_hub_download



def _limit(rows, n):    
    if not n or n >= len(rows):
        return rows
    rng = random.Random(config.SEED)
    indices = sorted(rng.sample(range(len(rows)), n))
    return [rows[i] for i in indices]


# ---------------- HumanEval ----------------
HUMANEVAL_REPO = "openai/openai_humaneval"


def load_humaneval():
    ds = load_dataset(HUMANEVAL_REPO, split="test")
    rows = [{"id": r["task_id"], "prompt": r["prompt"], "test": r["test"],
             "entry_point": r["entry_point"], "gold": r["canonical_solution"]} for r in ds]
    return _limit(rows, config.BENCHMARK_LIMIT)


# ---------------- MBPP ----------------
MBPP_REPO = "google-research-datasets/mbpp"


def load_mbpp():
    ds = load_dataset(MBPP_REPO, "sanitized", split="test")
    rows = []
    for r in ds:
        prompt = (f'"""\n{r["prompt"]}\n"""\n# Tests:\n' + "\n".join(r["test_list"]) + "\n")
        rows.append({"id": f"mbpp/{r['task_id']}", "prompt": prompt,
                     "tests": r["test_list"], "gold": r["code"]})
    return _limit(rows, config.BENCHMARK_LIMIT)


# ---------------- HumanEval-X (Python -> Java translation) ----------------
HUMANEVAL_X_REPO = "THUDM/humaneval-x"


def _hex_split(lang: str) -> list[dict]:
    """Load one HumanEval-X language split.

    `THUDM/humaneval-x` ships as a dataset *script* (`humaneval-x.py`) which newer
    `datasets` versions refuse to execute. We bypass `load_dataset()` and download the
    raw `.jsonl.gz` shard directly via `huggingface_hub`.
    """
    candidates = [
        f"data/{lang}/data/humaneval.jsonl",             # current THUDM layout (uncompressed)
        f"data/{lang}/data/humaneval_{lang}.jsonl.gz",   # original CodeGeeX layout
        f"data/{lang}/humaneval_{lang}.jsonl.gz",
        f"{lang}/data/humaneval_{lang}.jsonl.gz",
        f"data/{lang}/data/humaneval.jsonl.gz",          # in case it gets re-compressed
    ]
    last_err: Exception | None = None
    for filename in candidates:
        try:
            path = hf_hub_download(
                repo_id=HUMANEVAL_X_REPO,
                filename=filename,
                repo_type="dataset",
            )
            opener = gzip.open if filename.endswith(".gz") else open
            with opener(path, "rt", encoding="utf-8") as f:
                return [json.loads(line) for line in f if line.strip()]
        except Exception as e:
            last_err = e
            continue
    raise RuntimeError(
        f"Failed to load HumanEval-X split '{lang}' from {HUMANEVAL_X_REPO}. "
        f"Last error: {last_err!r}"
    )


def load_humaneval_x_py2java():
    py = _hex_split("python")
    ja = _hex_split("java")
    by_id_py = {r["task_id"].split("/")[-1]: r for r in py}
    by_id_ja = {r["task_id"].split("/")[-1]: r for r in ja}
    rows = []
    for k in sorted(by_id_ja.keys(), key=lambda x: int(x) if x.isdigit() else x):
        if k not in by_id_py:
            continue
        rp, rj = by_id_py[k], by_id_ja[k]
        rows.append({
            "id": f"hex/{k}",
            "python_source": rp["prompt"] + rp["canonical_solution"],
            "java_prompt": rj["prompt"],
            "java_test": rj["test"],
            "java_declaration": rj.get("declaration", rj["prompt"]),
            "java_reference": rj["prompt"] + rj["canonical_solution"],
        })
    return _limit(rows, config.BENCHMARK_LIMIT)


def _humaneval_hex_key(task_id: str) -> str:
    return task_id.split("/")[-1]


def load_hex_lookup() -> dict[str, dict]:
    """HumanEval-X rows keyed by problem number (e.g. '0', '1', ...)."""
    py = _hex_split("python")
    ja = _hex_split("java")
    by_id_py = {r["task_id"].split("/")[-1]: r for r in py}
    by_id_ja = {r["task_id"].split("/")[-1]: r for r in ja}
    lookup = {}
    for k in by_id_ja:
        if k not in by_id_py:
            continue
        rp, rj = by_id_py[k], by_id_ja[k]
        lookup[k] = {
            "hex_id": f"hex/{k}",
            "python_source": rp["prompt"] + rp["canonical_solution"],
            "java_prompt": rj["prompt"],
            "java_test": rj["test"],
            "java_reference": rj["prompt"] + rj["canonical_solution"],
            "entry_point_py": rp.get("entry_point"),
            "test_py": rp.get("test"),
        }
    return lookup


def load_humaneval_for_extension():
    """HumanEval rows with NL/PL1 fields and optional HEX java metadata."""

    hex_lookup = load_hex_lookup()
    rows = []
    for r in load_humaneval():
        key = _humaneval_hex_key(r["id"])
        hx = hex_lookup.get(key, {})
        rows.append({
            **r,
            "nl": extract_docstring_from_prompt(r["prompt"]),
            "pl1": r["prompt"] + r["gold"],
            "pl2": None,
            "hex_key": key,
            "java_test": hx.get("java_test"),
            "java_reference": hx.get("java_reference"),
        })
    return _limit(rows, config.EXTEND_LIMIT)


def load_mbpp_for_extension():

    rows = []
    for r in load_mbpp():
        rows.append({
            **r,
            "nl": extract_docstring_from_prompt(r["prompt"]) or r["prompt"],
            "pl1": r["gold"] if r["gold"].lstrip().startswith("def") else r["prompt"] + r["gold"],
            "pl2": None,
        })
    return _limit(rows, config.EXTEND_LIMIT)


def load_humaneval_x_for_extension():
    """HumanEval-X rows with PL1/PL2; NL to be generated."""
    rows = []
    for r in load_humaneval_x_py2java():
        rows.append({
            **r,
            "nl": None,
            "pl1": r["python_source"],
            "pl2": r["java_reference"],
            "java_declaration": r.get("java_declaration"),
        })
    return _limit(rows, config.EXTEND_LIMIT)




### 1.11 — Inline dataset extension


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

from tqdm.auto import tqdm



def _gen_pl2_from_nl_pl1(tokenizer, model, nl: str, pl1: str, discrepancy: str | None, pl2_attempt: str | None) -> str:
    if discrepancy and pl2_attempt:
        prompt = build_pl2_feedback_prompt(tokenizer, nl, pl1, pl2_attempt, discrepancy)
    else:
        prompt = build_pl2_generation_prompt(tokenizer, nl, pl1)
    raw = generate(tokenizer, model, prompt, config.DECODING_GREEDY)
    return extract_java_body(raw)


def _gen_nl_from_pl1_pl2(tokenizer, model, pl1: str, pl2: str, discrepancy: str | None, nl_attempt: str | None) -> str:
    if discrepancy and nl_attempt:
        prompt = build_nl_feedback_prompt(tokenizer, nl_attempt, pl1, pl2, discrepancy)
    else:
        prompt = build_nl_generation_prompt(tokenizer, pl1, pl2)
    raw = generate(tokenizer, model, prompt, config.DECODING_GREEDY)
    return raw.strip()


def _gen_pl1_from_nl(tokenizer, model, nl: str, humaneval_prompt_stub: str | None = None) -> str:
    if humaneval_prompt_stub:
        prompt = build_completion_prompt(tokenizer, {"prompt": humaneval_prompt_stub})
    else:
        system = "You are an expert Python programmer."
        user = (
            "Write a complete Python solution for this specification.\n"
            "Return the full program in a ```python ... ``` block.\n\n"
            f"{nl}"
        )
        msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    raw = generate(tokenizer, model, prompt, config.DECODING_GREEDY)
    return extract_python_body(raw)


def _gen_pl2_from_nl(tokenizer, model, nl: str, java_prompt_stub: str | None = None) -> str:
    if java_prompt_stub:
        prompt = build_java_completion_from_nl_prompt(tokenizer, java_prompt_stub)
    else:
        system = "You are an expert Java programmer."
        user = (
            "Write a complete Java solution (class Solution) for this specification.\n"
            "Return the full program in a ```java ... ``` block.\n\n"
            f"{nl}"
        )
        msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    raw = generate(tokenizer, model, prompt, config.DECODING_GREEDY)
    return extract_java_body(raw)


def extend_humaneval_row(row: dict, tokenizer, model, max_retries: int | None = None) -> dict[str, Any]:
    """Generate missing PL2 from NL+PL1; validate with Python+Java tests; retry up to N times."""
    max_retries = max_retries or config.EXTEND_MAX_RETRIES
    nl, pl1 = row["nl"], row["pl1"]
    pl2 = None
    discrepancy = None
    pl2_attempt = None
    validation = {}

    for attempt in range(1, max_retries + 1):
        pl2 = _gen_pl2_from_nl_pl1(tokenizer, model, nl, pl1, discrepancy, pl2_attempt)
        pl2_attempt = pl2
        validation = validate_pl2_against_pl1_humaneval(
            pl1, pl2,
            test=row["test"],
            entry_point=row["entry_point"],
            java_test=row.get("java_test"),
        )
        if validation["valid"]:
            break
        discrepancy = validation.get("error") or "validation failed"

    return {
        "dataset": "humaneval",
        "id": row["id"],
        "nl": nl,
        "pl1": pl1,
        "pl2": pl2,
        "pl2_valid": bool(validation.get("valid")),
        "pl2_attempts": attempt,
        "validation": validation,
        "hex_key": row.get("hex_key"),
    }


def extend_mbpp_row(row: dict, tokenizer, model, max_retries: int | None = None) -> dict[str, Any]:
    max_retries = max_retries or config.EXTEND_MAX_RETRIES
    nl, pl1 = row["nl"], row["pl1"]
    pl2 = None
    discrepancy = None
    pl2_attempt = None
    validation = {}

    for attempt in range(1, max_retries + 1):
        pl2 = _gen_pl2_from_nl_pl1(tokenizer, model, nl, pl1, discrepancy, pl2_attempt)
        pl2_attempt = pl2
        validation = validate_pl2_against_pl1_mbpp(pl1, pl2, row["tests"])
        if validation["valid"]:
            break
        discrepancy = validation.get("error") or "validation failed"

    return {
        "dataset": "mbpp",
        "id": row["id"],
        "nl": nl,
        "pl1": pl1,
        "pl2": pl2,
        "pl2_valid": bool(validation.get("valid")),
        "pl2_attempts": attempt,
        "validation": validation,
    }


def extend_humaneval_x_row(row: dict, tokenizer, model, max_retries: int | None = None) -> dict[str, Any]:
    """Generate missing NL from PL1+PL2; validate by NL→PL1' and NL→PL2'; retry up to N times."""
    max_retries = max_retries or config.EXTEND_MAX_RETRIES
    pl1, pl2 = row["pl1"], row["pl2"]
    nl = None
    discrepancy = None
    nl_attempt = None
    validation = {}

    for attempt in range(1, max_retries + 1):
        nl = _gen_nl_from_pl1_pl2(tokenizer, model, pl1, pl2, discrepancy, nl_attempt)
        nl_attempt = nl
        pl1_gen = _gen_pl1_from_nl(tokenizer, model, nl)
        pl2_gen = _gen_pl2_from_nl(tokenizer, model, nl, row.get("java_declaration"))

        key = row["id"].split("/")[-1]
        hx = load_hex_lookup().get(key, {})
        test_py = hx.get("test_py")
        entry_point = hx.get("entry_point_py")

        if test_py and entry_point:
            validation = validate_nl_via_regeneration(
                nl, pl1, pl2, pl1_gen, pl2_gen,
                test=test_py,
                entry_point=entry_point,
                java_test=row.get("java_test"),
            )
        else:
            pl1_ast, pl1_ast_err = python_ast_match(pl1, pl1_gen)
            jr = run_java(pl2_gen, row["java_test"])
            validation = {
                "valid": pl1_ast and jr["passed"],
                "pl1_ast_match": pl1_ast,
                "pl2_java_pass": jr["passed"],
                "error": pl1_ast_err or jr.get("error"),
            }

        if validation.get("valid"):
            break
        discrepancy = validation.get("error") or "nl regeneration validation failed"

    return {
        "dataset": "humaneval_x",
        "id": row["id"],
        "nl": nl,
        "pl1": pl1,
        "pl2": pl2,
        "nl_valid": bool(validation.get("valid")),
        "nl_attempts": attempt,
        "validation": validation,
    }


def run_extension(
    tokenizer,
    model,
    *,
    humaneval_rows: list[dict] | None = None,
    mbpp_rows: list[dict] | None = None,
    hex_rows: list[dict] | None = None,
) -> list[dict[str, Any]]:

    humaneval_rows = humaneval_rows if humaneval_rows is not None else load_humaneval_for_extension()
    mbpp_rows = mbpp_rows if mbpp_rows is not None else load_mbpp_for_extension()
    hex_rows = hex_rows if hex_rows is not None else load_humaneval_x_for_extension()

    out: list[dict[str, Any]] = []
    for row in tqdm(humaneval_rows, desc="extend humaneval (NL+PL1→PL2)"):
        out.append(extend_humaneval_row(row, tokenizer, model))
    for row in tqdm(mbpp_rows, desc="extend mbpp (NL+PL1→PL2)"):
        out.append(extend_mbpp_row(row, tokenizer, model))
    for row in tqdm(hex_rows, desc="extend humaneval-x (PL1+PL2→NL)"):
        out.append(extend_humaneval_x_row(row, tokenizer, model))
    return out


def save_extended(rows: list[dict[str, Any]], path: Path | None = None) -> Path:
    config.ensure_dirs()
    path = path or config.EXTENDED_DIR / "unified_dataset.jsonl"
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    return path


def summarize_extension(rows: list[dict[str, Any]]) -> str:
    from collections import defaultdict

    stats = defaultdict(lambda: {"total": 0, "valid": 0})
    for r in rows:
        ds = r["dataset"]
        stats[ds]["total"] += 1
        if r.get("pl2_valid") or r.get("nl_valid"):
            stats[ds]["valid"] += 1
    lines = ["dataset  total  valid  rate"]
    for ds, s in sorted(stats.items()):
        rate = s["valid"] / s["total"] if s["total"] else 0
        lines.append(f"{ds:12s} {s['total']:5d} {s['valid']:5d} {rate:5.1%}")
    return "\n".join(lines)


In [ ]:
import types

models = types.SimpleNamespace(
    load_qwen=load_qwen,
    unload=unload,
    setup_java_runtime=setup_java_runtime,
)
prompts = types.SimpleNamespace(
    build_completion_prompt=build_completion_prompt,
    build_translation_prompt=build_translation_prompt,
    generate=generate,
    extract_python_body=extract_python_body,
    extract_java_body=extract_java_body,
    build_pl2_generation_prompt=build_pl2_generation_prompt,
    build_pl2_feedback_prompt=build_pl2_feedback_prompt,
    build_nl_generation_prompt=build_nl_generation_prompt,
    build_nl_feedback_prompt=build_nl_feedback_prompt,
    build_java_completion_from_nl_prompt=build_java_completion_from_nl_prompt,
)
execution = types.SimpleNamespace(
    run_python=run_python,
    run_java=run_java,
    compile_java=compile_java,
    smoke_test_java=smoke_test_java,
)
print('Inline library ready.')


In [ ]:
# 1.12 — runtime settings + Java smoke test
import os

os.environ.setdefault('CODEGEN_SEED', '42')
os.environ.setdefault('CODEGEN_SAMPLE', '50')
# os.environ['CODEGEN_SMOKE'] = '1'

refresh_config_limits()
config.refresh_config_limits()
ensure_dirs()
print('Smoke mode:  ', config.SMOKE)
print('Problem cap: ', config.BENCHMARK_LIMIT)

setup_java_runtime()
result = smoke_test_java()
assert result['passed'], result
print('Java sandbox OK:', result)


## K=0 baselines


In [ ]:
# 2.0 — clear GPU memory
import gc
import pandas as pd
import torch
from tqdm.auto import tqdm

gc.collect()
torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f'GPU free: {free/1e9:.2f} / {total/1e9:.2f} GB')


### 2.2 Per-benchmark scorers


In [ ]:
def score_humaneval(row, generated):
    program = prompts.extract_python_body(generated)
    full = program + "\n\n" + row['test'] + f"\n\ncheck({row['entry_point']})\n"
    res = execution.run_python(full)
    return {'score': 1.0 if res['passed'] else 0.0, 'passed': res['passed'], 'error': res['error']}

def score_mbpp(row, generated):
    program = prompts.extract_python_body(generated)
    full = program + "\n\n" + "\n".join(row['tests']) + "\n"
    res = execution.run_python(full)
    return {'score': 1.0 if res['passed'] else 0.0, 'passed': res['passed'], 'error': res['error']}

def score_humaneval_x(row, generated):
    java_source = prompts.extract_java_body(generated)
    res = execution.run_java(java_source, row['java_test'])
    return {'score': 1.0 if res['passed'] else 0.0, 'passed': res['passed'], 'error': res['error']}


### 2.3 Generate + score one benchmark


In [ ]:
def run_benchmark(model_size, tokenizer, model, benchmark_name, rows):
    out = []
    for row in tqdm(rows, desc=f'{model_size} | {benchmark_name}'):
        if benchmark_name in ('humaneval', 'mbpp'):
            prompt = prompts.build_completion_prompt(tokenizer, {'prompt': row['prompt']})
        elif benchmark_name == 'humaneval_x_py2java':
            prompt = prompts.build_translation_prompt(tokenizer, row['python_source'],
                                                      row['java_declaration'])
        else:
            raise ValueError(benchmark_name)
        gen = prompts.generate(tokenizer, model, prompt, config.DECODING_GREEDY)
        if benchmark_name == 'humaneval':
            r = score_humaneval(row, gen)
        elif benchmark_name == 'mbpp':
            r = score_mbpp(row, gen)
        else:
            r = score_humaneval_x(row, gen)
        out.append({'model': model_size, 'benchmark': benchmark_name, 'problem_id': row['id'],
                    'generated': gen[:2000], **r})
    return out


In [ ]:
benches = {
    'humaneval': load_humaneval(),
    'mbpp': load_mbpp(),
    'humaneval_x_py2java': load_humaneval_x_py2java(),
}
for name, rows in benches.items():
    print(f'{name}: {len(rows)} problems')


### 2.4 Run Qwen-1.5B


In [ ]:
def _run_at_size(size, benches):
    tok, model = models.load_qwen(size)
    out = []
    try:
        for bn, rows in benches.items():
            if rows:
                out.extend(run_benchmark(size, tok, model, bn, rows))
    finally:
        del tok, model
        gc.collect()
        torch.cuda.empty_cache()
    return out

results_15 = _run_at_size('1.5b', benches)
print(f'{len(results_15)} rows from 1.5B')


### 2.5 Run Qwen-7B (4-bit)


In [ ]:
tok7, m7 = models.load_qwen('7b')
results_7 = []
for bn, rows in benches.items():
    if rows:
        results_7.extend(run_benchmark('7b', tok7, m7, bn, rows))
models.unload(m7)
print(f'{len(results_7)} rows from 7B')


In [ ]:
all_rows = results_15 + results_7


### 2.6 Save + summarize


In [ ]:
df = pd.DataFrame(all_rows)
out_path = config.RESULTS_DIR / 'baselines.csv'
df.to_csv(out_path, index=False)
print(f'Saved {len(df)} rows to {out_path}')
summary = df.groupby(['model', 'benchmark'])['score'].agg(['count', 'mean']).reset_index()
summary.columns = ['model', 'benchmark', 'n', 'mean_score']
print(summary.to_string(index=False))


## Part 3 — Mentor dataset extension (NL + PL1 + PL2)


In [ ]:
# 3.1 — extension settings
import os
import gc
import torch

os.environ.setdefault('CODEGEN_EXTEND_SAMPLE', '5')
os.environ.setdefault('CODEGEN_EXTEND_RETRIES', '3')
refresh_config_limits()
config.refresh_config_limits()
ensure_dirs()

he_ext = load_humaneval_for_extension()
mbpp_ext = load_mbpp_for_extension()
hex_ext = load_humaneval_x_for_extension()
print('Extension rows:', len(he_ext), len(mbpp_ext), len(hex_ext))
print('Output dir:', config.EXTENDED_DIR)


In [ ]:
# 3.2 — load extension model
gc.collect()
torch.cuda.empty_cache()
EXTEND_MODEL_SIZE = os.environ.get('CODEGEN_EXTEND_MODEL', '1.5b')
tok_ext, model_ext = models.load_qwen(EXTEND_MODEL_SIZE)
print('Extension model:', EXTEND_MODEL_SIZE)


In [ ]:
# 3.3 — preview rows
print('HumanEval:', he_ext[0]['id'], '| NL:', he_ext[0]['nl'][:80], '...')
print('MBPP:', mbpp_ext[0]['id'])
print('HumanEval-X:', hex_ext[0]['id'])


In [ ]:
# 3.4 — run extension
extended_rows = run_extension(tok_ext, model_ext, humaneval_rows=he_ext,
                              mbpp_rows=mbpp_ext, hex_rows=hex_ext)
print(summarize_extension(extended_rows))


In [ ]:
# 3.5 — save
out_jsonl = save_extended(extended_rows)
print('Saved:', out_jsonl)
df_ext = pd.DataFrame(extended_rows)
print(df_ext[['dataset', 'id', 'pl2_valid', 'nl_valid']].head(10).to_string(index=False))
del tok_ext, model_ext
gc.collect()
torch.cuda.empty_cache()
